#  Objectif :
>
> 1. Construire une matrice BoW (Bag-of-Words) pour un corpus de 37 textes.
> 2. Calculer la matrice des distances entre tous les textes.
> 3. Calculer, pour chaque texte, la distance au **centroïde** (moyenne) de chaque **genre** (ex. *Aventure*, *Récit de voyage*, *Amour*).
> 4. **Bonus** : visualiser les distances (heatmap, MDS/TSNE).
>## 📦 Pré-requis

* Python ≥ 3.9
* `pandas`, `numpy`, `scikit-learn`, `matplotlib`, `unidecode` (facultatif)
* Arborescence (adapter `CORPUS_DIR` si besoin) :

```
~/Documents/cours/2025/TALL_CPES_25-26/seance_04/TD/corpus/*.txt



```



## 🗂️ 0. Imports & paramètres


In [2]:
!pip install scikit-learn

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 62.0/62.0 kB 1.0 MB/s eta 0:00:00 MB/s eta 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.5/9.5 MB 11.7 MB/s eta 0:00:00m eta 0:00:010:01:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 308.4/308.4 kB 9.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 35.7/35.7 MB 17.0 MB/s eta 0:00:00m eta 0:00:010:00:01
Note: you may need to restart the kernel to use updated packages.


In [1]:
import os, glob, re, json, textwrap
from pathlib import Path

import numpy as np
import pandas as pd

from sklearn.feature_extraction.text import CountVectorizer
from sklearn.metrics import pairwise_distances
from sklearn.preprocessing import normalize

import matplotlib.pyplot as plt

# (Optionnel) pour normaliser les accents dans les tokens
try:
    from unidecode import unidecode
    USE_UNIDECODE = True
except Exception:
    USE_UNIDECODE = False

In [22]:

CORPUS_DIR = Path.home() / "Documents/cours/2025/TALL_CPES_25-26/seance_04/TD/corpus"

# --- Paramètres BoW
MAX_FEATURES = 1000     # nb max de mots (les plus fréquents)
MIN_DF = 2              # mot conservé s'il apparaît dans >= MIN_DF documents
STOP_LANG = "english"    # stopwords intégrés de scikit-learn ("french", None, etc.)
LOWERCASE = True
NGRAM_RANGE = (1, 1)    # unigrams; vous pouvez tester (1,2)

## 📥 1. Charger les textes

> On lit tous les `.txt` du dossier, en gérant au mieux l’encodage.


In [7]:
# 1) Load corpus files
paths = sorted(glob.glob(str(CORPUS_DIR / "*.txt")))

def read_text(path):
    # essai multiprotocole d'encodage
    for enc in ("utf-8", "latin-1", "cp1252"):
        try:
            with open(path, "r", encoding=enc, errors="ignore") as f:
                content = f.read()
            return content
        except Exception:
            continue
    raise IOError(f"Cannot read {path} with common encodings.")

docs = [read_text(p) for p in paths]
filenames = [os.path.basename(p) for p in paths]



In [8]:
len(paths), filenames[:5]

(37,
 ['1900_Verne-Jules-_Seconde-patrie.txt',
  '1901_Thurner-Georges_Mademoiselle-Flammette.txt',
  '1902_Allais-Alphonse_Le-Captain-Cap.txt',
  '1902_Le-Rouge-Gustave_La-Princesse-des-airs_Tome-I.txt',
  '1902_Le-Rouge-Gustave_La-Princesse-des-airs_Tome-II.txt'])

# 2) Genre mapping (à adapter)

In [11]:

CSV_PATH = CORPUS_DIR.parent / "genres.csv"  # placez un CSV à ../genres.csv si vous préférez

if CSV_PATH.exists():
    df_genres = pd.read_csv(CSV_PATH)  # colonnes: filename, genre
else:
    # Exemple: complétez ce dictionnaire pendant le TP
    manual_map = {
        # "1900_Verne-Jules-_Seconde-patrie.txt": "Aventure",
        # "1963_Bouvier-Nicolas_L-usage-du-monde.txt": "Récit de voyage",
        # "1932_Veuzit-Max-du_Petite-comtesse.txt": "Amour",
        # ...
    }
    df_genres = pd.DataFrame({
        "filename": filenames,
        "genre": [manual_map.get(fn, "Inconnu") for fn in filenames]
    })

# Sécurise l'ordre (aligné aux fichiers lus)
df_genres = df_genres.set_index("filename").loc[filenames].reset_index()
genres = df_genres["genre"].values
unique_genres = sorted(pd.unique(genres))
display(df_genres.head())
print("Genres:", unique_genres)


,filename,genre
0,1900_Verne-Jules-_Seconde-patrie.txt,Inconnu
1,1901_Thurner-Georges_Mademoiselle-Flammette.txt,Inconnu
2,1902_Allais-Alphonse_Le-Captain-Cap.txt,Inconnu
3,1902_Le-Rouge-Gustave_La-Princesse-des-airs_To...,Inconnu
4,1902_Le-Rouge-Gustave_La-Princesse-des-airs_To...,Inconnu


Genres: ['Inconnu']



## 🧱 3. Fonctions utilitaires : distances & BoW

> Nous vous **donnons** ci-dessous :
>
> * des définitions simples des **distances** (Cosine & Euclidienne)
> * une fonction pour construire le **DataFrame BoW** : N lignes = N romans ; M colonnes = M mots.


In [17]:
# 3) Regex tokenizer

TOKEN_REGEX = re.compile(r"[A-Za-zÀ-ÖØ-öø-ÿ]+(?:[-'][A-Za-zÀ-ÖØ-öø-ÿ]+)*")

def regex_tokenizer(text: str, lower: bool = True):
    if lower:
        text = text.lower()
    return TOKEN_REGEX.findall(text)


In [27]:
# 4) BoW avec le tokenizer regex 

def build_relfreq_dataframe_regex(texts,
                                  max_features=MAX_FEATURES,
                                  min_df=MIN_DF,
                                  stop_lang=STOP_LANG,
                                  ngram_range=NGRAM_RANGE):
    """
    Retourne:
      - df_relfreq: DataFrame (n_docs x n_terms) avec f_ij / sum_j f_ij
      - vectorizer: CountVectorizer entraîné
      - row_lengths: longueurs (somme des comptes) par doc (après tokenisation/stopwords)
    """
    vectorizer = CountVectorizer(
        tokenizer=lambda s: regex_tokenizer(s, lower=True),
        token_pattern=None,            # IMPORTANT: sinon le tokenizer custom est ignoré
        preprocessor=None,
        lowercase=False,               # déjà fait dans tokenizer
        strip_accents="unicode",       # normalise les accents
        stop_words=stop_lang,
        max_features=max_features,
        min_df=min_df,
        ngram_range=ngram_range
    )
    X = vectorizer.fit_transform(texts)            # sparse counts
    vocab = vectorizer.get_feature_names_out()
    counts = X.toarray().astype(float)             # dense pour la normalisation

    row_sums = counts.sum(axis=1, keepdims=True)   # longueurs après filtrage
    # Sécurité: éviter division par zéro (document vide après stopwords)
    zero_rows = (row_sums == 0).flatten()
    if np.any(zero_rows):
        # On laisse ces lignes à zéro; cosine_distance échouera si vecteur nul
        # mais c'est informatif pour les étudiants
        print(f"Attention: {zero_rows.sum()} document(s) avec longueur 0 après filtrage.")

    relfreq = np.divide(counts, row_sums, where=row_sums!=0)

    df_relfreq = pd.DataFrame(relfreq, index=filenames, columns=vocab)
    return df_relfreq, vectorizer, row_sums.flatten()

> 💡 *Remarque* : `strip_accents="unicode"` suffit en général. Si vous préférez un contrôle fin, vous pouvez pré-nettoyer avec `unidecode` avant vectorisation.
>
## 🧮 4. Construire la matrice BoW


In [29]:

df_bow, vect = build_relfreq_dataframe_regex(docs)
df_bow.iloc[:5, :10]


ValueError: too many values to unpack (expected 2)

In [26]:
df_bow.head()

,a,a-t-il,abord,absolument,aeroscaphe,affaire,affaires,affection,afin,age,...,voyant,voyez,voyons,vrai,vraiment,vu,vue,y,yeux,yvon
1900_Verne-Jules-_Seconde-patrie.txt,3448,18,50,14,0,9,6,9,72,19,...,7,4,11,67,1,11,93,537,41,0
1901_Thurner-Georges_Mademoiselle-Flammette.txt,1458,9,13,12,0,9,11,12,15,8,...,22,7,2,14,9,14,25,122,62,0
1902_Allais-Alphonse_Le-Captain-Cap.txt,960,1,7,1,0,9,2,0,5,6,...,0,11,3,19,9,24,11,79,17,0
1902_Le-Rouge-Gustave_La-Princesse-des-airs_Tome-I.txt,1930,10,47,17,155,6,6,7,10,21,...,10,6,0,11,16,21,26,156,55,135
1902_Le-Rouge-Gustave_La-Princesse-des-airs_Tome-II.txt,1963,6,28,17,80,8,4,0,5,4,...,6,3,0,12,6,16,21,170,52,112


## 📏 5. Matrice des distances entre textes

> Choisissez une métrique : **cosine** (recommandée pour du texte) ou **euclidienne** (sur BoW bruts).


In [18]:
# 5) Distance functions (exactement comme spécifié)

def euclidean_distance(a, b):
    return np.sqrt(np.sum((a - b) ** 2))

def vector_len(v):
    return np.sqrt(np.sum(v ** 2))

def cosine_distance(a, b):
    return 1 - np.dot(a, b) / (vector_len(a) * vector_len(b))

## 📊 7. Bonus — Visualisations


# 7.1) Heatmap (matplotlib)
plt.figure(figsize=(8, 7))
plt.imshow(df_D_cos, interpolation="nearest")


